# 06 · Supervised Modeling

Train 5 classifiers + 2 regressors with StratifiedGroupKFold (groups = `disaster_number`) and SMOTE in an imblearn pipeline. Tune RF with GridSearchCV, XGB with Optuna. Evaluate on the VAL split.

In [1]:
import sys, os
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', 60)
from config import (
    DATA_PATHS, HURRICANE_META, STATES_IN_SCOPE,
    TARGET_COL, TARGET_CLASS_COL, FEATURE_GROUPS,
    RANDOM_STATE, SEVERITY_BINS, SEVERITY_LABELS,
)
RAW = DATA_PATHS['raw']; INTERIM = DATA_PATHS['interim']
PROC = DATA_PATHS['processed']; MODELS = DATA_PATHS['models']
OUT = DATA_PATHS['outputs']


In [2]:
import joblib
from sklearn.preprocessing import LabelEncoder
from src.modeling import (
    build_preprocessor, build_pipeline, get_classifiers, get_regressors,
    cv_score, tune_rf_grid, tune_xgb_optuna, regression_eval, classification_eval,
)
from config import CONTINUOUS_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES
df = pd.read_csv(PROC / 'abt_with_clusters.csv', dtype={'zip_code': str}).dropna(subset=[TARGET_CLASS_COL])
features = (FEATURE_GROUPS['demographics']+FEATURE_GROUPS['svi']
    +FEATURE_GROUPS['food_access']+FEATURE_GROUPS['flood']
    +FEATURE_GROUPS['storm']+['cluster_label','state'])
features = [f for f in features if f in df.columns]
X, y, g = df[features], df[TARGET_CLASS_COL], df['disaster_number']
le = LabelEncoder().fit(SEVERITY_LABELS); y_enc = le.transform(y)
tr = df['train_test_split']=='TRAIN'; va = df['train_test_split']=='VAL'

# Filter preprocessor feature lists to columns actually present in X
avail = set(X.columns)
cont = [c for c in CONTINUOUS_FEATURES if c in avail]
if 'cluster_label' in avail and 'cluster_label' not in cont:
    cont.append('cluster_label')
binf = [c for c in BINARY_FEATURES if c in avail]
catf = [c for c in CATEGORICAL_FEATURES if c in avail]
pre = build_preprocessor(continuous=cont, categorical=catf, binary=binf)
print('using', len(cont), 'cont +', len(catf), 'cat +', len(binf), 'bin features')

using 22 cont + 2 cat + 1 bin features


## Cross-val on TRAIN split

In [3]:
results = {}
for name, clf in get_classifiers().items():
    pipe = build_pipeline(clf, preprocessor=pre)
    mean, arr = cv_score(pipe, X[tr], pd.Series(y_enc[tr]), g[tr])
    results[name] = mean
    print(f'{name:6s}  CV F1-weighted = {mean:.3f}  (folds: {arr.round(3)})')

C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


rf      CV F1-weighted = 0.495  (folds: [0.584 0.607 0.229 0.65  0.402])


C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


svm     CV F1-weighted = 0.403  (folds: [0.246 0.61  0.343 0.571 0.245])


C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


nb      CV F1-weighted = 0.357  (folds: [0.287 0.429 0.375 0.472 0.219])


C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWa

lr      CV F1-weighted = 0.414  (folds: [0.496 0.542 0.259 0.259 0.514])


C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


xgb     CV F1-weighted = 0.479  (folds: [0.428 0.596 0.16  0.619 0.594])


## Tune RF + XGBoost

In [4]:
rf_gs = tune_rf_grid(X[tr], pd.Series(y_enc[tr]), g[tr], preprocessor=pre)
print('RF best:', rf_gs.best_params_, rf_gs.best_score_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
RF best: {'model__max_depth': 20, 'model__min_samples_leaf': 1, 'model__n_estimators': 400} 0.5074459094612163


In [5]:
try:
    study = tune_xgb_optuna(X[tr], pd.Series(y_enc[tr]), g[tr], n_trials=50, preprocessor=pre)
    print('XGB best:', study.best_params, study.best_value)
except Exception as e:
    print('XGB tuning skipped:', e); study = None

[I 2026-05-03 19:09:00,900] A new study created in memory with name: no-name-26fb719f-fa83-4f5f-9e30-d09ab58db035


  0%|          | 0/50 [00:00<?, ?it/s]

C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


[I 2026-05-03 19:09:18,553] Trial 0 finished with value: 0.49665551838126987 and parameters: {'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746}. Best is trial 0 with value: 0.49665551838126987.


C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


...

[I 2026-05-03 19:21:40,342] Trial 47 finished with value: 0.5004742003790972 and parameters: {'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.023530574443084117, 'subsample': 0.7044177431578483, 'colsample_bytree': 0.8664466438168699}. Best is trial 27 with value: 0.5284286620909786.


C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


[I 2026-05-03 19:22:24,350] Trial 48 finished with value: 0.4997814851366641 and parameters: {'n_estimators': 400, 'max_depth': 8, 'learning_rate': 0.011465272191239132, 'subsample': 0.9380443273863568, 'colsample_bytree': 0.7786178771316112}. Best is trial 27 with value: 0.5284286620909786.


C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


[I 2026-05-03 19:22:47,780] Trial 49 finished with value: 0.5167007280668178 and parameters: {'n_estimators': 150, 'max_depth': 9, 'learning_rate': 0.014732355057864743, 'subsample': 0.9984033158093563, 'colsample_bytree': 0.8198440956468087}. Best is trial 27 with value: 0.5284286620909786.
XGB best: {'n_estimators': 100, 'max_depth': 8, 'learning_rate': 0.010363848712718024, 'subsample': 0.9506404929898924, 'colsample_bytree': 0.8266600795415145} 0.5284286620909786


## Fit on TRAIN, evaluate on VAL — WITH cluster_label

In [6]:
val_scores = {}
for name, clf in get_classifiers().items():
    pipe = build_pipeline(clf, preprocessor=pre).fit(X[tr], y_enc[tr])
    pred = pipe.predict(X[va])
    val_scores[name+'_withcluster'] = classification_eval(y_enc[va], pred)['f1_weighted']
    print(f'{name} WITH cluster  F1={val_scores[name+"_withcluster"]:.3f}')

C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


rf WITH cluster  F1=0.400


C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


svm WITH cluster  F1=0.383


C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


nb WITH cluster  F1=0.329


C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


lr WITH cluster  F1=0.319
xgb WITH cluster  F1=0.409


C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## Same evaluation WITHOUT cluster_label

In [7]:
features_noc = [f for f in features if f != 'cluster_label']
cont_noc = [c for c in cont if c != 'cluster_label' and c in features_noc]
pre_noc = build_preprocessor(continuous=cont_noc, categorical=catf, binary=binf)
for name, clf in get_classifiers().items():
    pipe = build_pipeline(clf, preprocessor=pre_noc).fit(X[tr][features_noc], y_enc[tr])
    pred = pipe.predict(X[va][features_noc])
    val_scores[name+'_nocluster'] = classification_eval(y_enc[va], pred)['f1_weighted']
cmp = pd.DataFrame({
  'with_cluster': {k.replace('_withcluster',''): v for k,v in val_scores.items() if k.endswith('_withcluster')},
  'without_cluster': {k.replace('_nocluster',''): v for k,v in val_scores.items() if k.endswith('_nocluster')},
}); cmp['delta'] = cmp['with_cluster'] - cmp['without_cluster']; cmp

C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning,

,with_cluster,without_cluster,delta
rf,0.400001,0.393783,0.006219
svm,0.383389,0.389286,-0.005898
nb,0.329232,0.339917,-0.010685
lr,0.318542,0.320907,-0.002365
xgb,0.408765,0.421378,-0.012614


## Regression variants

In [8]:
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.compose import TransformedTargetRegressor
# Heavy right-tailed target → predict log1p(y), invert with expm1.
# Metrics are still computed on the original scale.
regs_scores = {}
for name, reg in get_regressors().items():
    inner = SkPipeline([('pre', build_preprocessor(continuous=cont, categorical=catf, binary=binf)),
                        ('reg', reg)])
    pipe_r = TransformedTargetRegressor(regressor=inner, func=np.log1p, inverse_func=np.expm1)
    pipe_r.fit(X[tr], df.loc[tr, TARGET_COL])
    pred = pipe_r.predict(X[va])
    regs_scores[name] = regression_eval(df.loc[va, TARGET_COL], pred)
regs_scores

C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


{'rf': {'rmse': 111.00579993928186,
  'mae': 31.36478296774866,
  'r2': 0.05442003418837127},
 'xgb': {'rmse': 113.70974787759815,
  'mae': 29.441192305312715,
  'r2': 0.007792938194153054}}

## Save best classifier + regressor

In [9]:
best_name = max({k:v for k,v in val_scores.items() if k.endswith('_withcluster')}.items(), key=lambda kv: kv[1])[0].replace('_withcluster','')
print('best classifier:', best_name)
best_clf = build_pipeline(get_classifiers()[best_name], preprocessor=pre).fit(X[tr], y_enc[tr])
joblib.dump({'pipe': best_clf, 'label_encoder': le, 'features': features},
            MODELS / 'best_classifier.pkl')
best_reg_name = min(regs_scores.items(), key=lambda kv: kv[1]['rmse'])[0]
print('best regressor:', best_reg_name)
inner = SkPipeline(
    [('pre', build_preprocessor(continuous=cont, categorical=catf, binary=binf)),
     ('reg', get_regressors()[best_reg_name])])
best_reg = TransformedTargetRegressor(regressor=inner, func=np.log1p, inverse_func=np.expm1)
best_reg.fit(X[tr], df.loc[tr, TARGET_COL])
joblib.dump({'pipe': best_reg, 'features': features}, MODELS / 'best_regressor.pkl')
print('saved', MODELS / 'best_classifier.pkl', '/', MODELS / 'best_regressor.pkl')

best classifier: xgb
best regressor: rf
saved C:\Users\chait\OneDrive\Documents\ML_DATA_245\hurricane-food-relief\models\best_classifier.pkl / C:\Users\chait\OneDrive\Documents\ML_DATA_245\hurricane-food-relief\models\best_regressor.pkl
